# Sentiment Analysis using Machine Learning

Binary sentiment classification using NLP, TF-IDF, Logistic Regression, and Multinomial Naive Bayes.

## 1. Import Libraries

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")

## 2. Load the Dataset

In [ ]:
def load_sentences(path, label):
    with open(path, "r", encoding="latin-1") as file:
        return [(line.strip(), label) for line in file if line.strip()]

positive = load_sentences("data/positive.txt", "positive")
negative = load_sentences("data/negative.txt", "negative")

df = pd.DataFrame(positive + negative, columns=["text", "label"])
print("Dataset shape:", df.shape)
df.head()

## 3. Exploratory Data Analysis

In [ ]:
print(df["label"].value_counts())

plt.figure(figsize=(6, 4))
df["label"].value_counts().reindex(["positive", "negative"]).plot(kind="bar")
plt.title("Sentiment Distribution")
plt.xlabel("Sentiment")
plt.ylabel("Number of Samples")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 4. Text Preprocessing

In [ ]:
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    text = text.lower()
    text = re.sub(r"[^a-z\s]", " ", text)
    tokens = text.split()
    tokens = [word for word in tokens if word not in stop_words]
    tokens = [lemmatizer.lemmatize(word) for word in tokens]
    return " ".join(tokens)

df["clean_text"] = df["text"].apply(preprocess_text)
df[["text", "clean_text", "label"]].head()

## 5. Train-Test Split

In [ ]:
X_train_text, X_test_text, y_train, y_test = train_test_split(
    df["clean_text"],
    df["label"],
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

print("Training samples:", len(X_train_text))
print("Testing samples:", len(X_test_text))

## 6. TF-IDF Feature Extraction

In [ ]:
vectorizer = TfidfVectorizer(max_features=5000)

X_train = vectorizer.fit_transform(X_train_text)
X_test = vectorizer.transform(X_test_text)

print("Training feature matrix:", X_train.shape)
print("Testing feature matrix:", X_test.shape)

## 7. Logistic Regression

In [ ]:
logistic_model = LogisticRegression(max_iter=1000)
logistic_model.fit(X_train, y_train)

logistic_pred = logistic_model.predict(X_test)

print("Logistic Regression Accuracy:",
      accuracy_score(y_test, logistic_pred))
print("\nClassification Report:")
print(classification_report(y_test, logistic_pred))

## 8. Multinomial Naive Bayes

In [ ]:
nb_model = MultinomialNB()
nb_model.fit(X_train, y_train)

nb_pred = nb_model.predict(X_test)

print("Multinomial Naive Bayes Accuracy:",
      accuracy_score(y_test, nb_pred))
print("\nClassification Report:")
print(classification_report(y_test, nb_pred))

## 9. Model Comparison

In [ ]:
results = pd.DataFrame({
    "Model": ["Logistic Regression", "Multinomial Naive Bayes"],
    "Accuracy": [
        accuracy_score(y_test, logistic_pred),
        accuracy_score(y_test, nb_pred)
    ]
})

results

## 10. Confusion Matrix

In [ ]:
cm = confusion_matrix(
    y_test, logistic_pred, labels=["negative", "positive"]
)

plt.figure(figsize=(6, 5))
plt.imshow(cm)
plt.title("Logistic Regression Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.xticks([0, 1], ["negative", "positive"])
plt.yticks([0, 1], ["negative", "positive"])

for i in range(2):
    for j in range(2):
        plt.text(j, i, cm[i, j], ha="center", va="center")

plt.colorbar()
plt.tight_layout()
plt.show()

## 11. Custom Predictions

In [ ]:
def predict_sentiment(text):
    cleaned = preprocess_text(text)
    features = vectorizer.transform([cleaned])
    prediction = logistic_model.predict(features)[0]
    return prediction

examples = [
    "This movie was amazing",
    "I hate this product",
    "Not bad but could be better"
]

for text in examples:
    print(f"{text} -> {predict_sentiment(text)}")

## 12. Conclusion

The project demonstrates a complete classical NLP pipeline for binary sentiment classification. Text is cleaned and transformed into TF-IDF features, then classified using Logistic Regression and Multinomial Naive Bayes. Model performance is evaluated on unseen test data using accuracy, precision, recall, F1-score, and a confusion matrix.